In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import PIL
from PIL import Image

import os
from os.path import join, exists
from tqdm import tqdm

seed = 42
np.random.seed(seed)

In [44]:
from google.colab import drive
drive.mount('/content/drive')

# get data and unzip (if not already)
if not exists('data_cleaned'):
  !unzip "/content/drive/My Drive/aps360/project_data"
else:
  print('Data already loaded')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data already loaded


In [45]:
BASE_DATA_PATH = 'data_cleaned'
IMAGE_PATH = join(BASE_DATA_PATH, 'images')
ANNOT_PATH = join(BASE_DATA_PATH, 'annotations')

In [46]:
df = pd.read_csv(join(ANNOT_PATH, 'master_alternate_augmented.csv'))
n = len(df)
value_counts = df['label'].value_counts()
print(f"Total Samples: {n}")
print(value_counts)


Total Samples: 21747
label
no finding           7932
pneumonia            4722
other                1477
heart disease        1088
infiltration         1088
consolidation        1088
effusion             1088
bronchiolitis        1088
bronchitis           1088
broncho-pnuemonia    1088
Name: count, dtype: int64


In [47]:
from skimage import io
from skimage.transform import resize
from copy import deepcopy

df_new = deepcopy(df)
# DESIRED_IMG_SIZE = (520,400)
DESIRED_IMG_SIZE = (130, 100)

save_folder = 'data_small_2'
if not exists(save_folder):
  os.makedirs(save_folder)

for i in tqdm(range(len(df))):
  row = df.iloc[i]
  fname = row['filename']
  # load image
  img_path = os.path.join(IMAGE_PATH, fname)
  img = Image.open(img_path).convert('L')
  img = np.array(img)
  scaled_x = DESIRED_IMG_SIZE[0] / img.shape[0]
  scaled_y = DESIRED_IMG_SIZE[1] / img.shape[1]
  if not exists(join(save_folder, fname)):
    # resize
    img_resized = resize(img, DESIRED_IMG_SIZE, anti_aliasing=True)
    img_resized = (img_resized * 255).astype(np.uint8)
    # save
    save_path = join(save_folder, fname)
    Image.fromarray(img_resized).save(save_path)

  # update bbox in df_new if needed
  if row['has bbox']:
    df_new.at[i, 'min_x'] = row['min_x'] * scaled_x
    df_new.at[i, 'min_y'] = row['min_y'] * scaled_y
    df_new.at[i, 'width'] = row['width'] * scaled_x
    df_new.at[i, 'height'] = row['height'] * scaled_y

100%|██████████| 21747/21747 [00:28<00:00, 751.64it/s] 


In [48]:
IMAGE_PATH = join(save_folder)

In [50]:
# only keep data with bboxes
n_tot = len(df)
df = df_new.loc[df_new['has bbox'] == 1]
# remove min_x that has nan rows
min_x_nan = df['min_x'].isna()
df = df.loc[~min_x_nan]
# print stats
n_bbox = len(df)
print(f"Total Samples: {n_tot}")
print(f"Samples with bbox: {n_bbox}")


# load images and bbox
images = []
bboxes = []
labels = []

for i in tqdm(range(len(df))):
    row = df.iloc[i]
    fname = row['filename']
    # image
    img_path = join(IMAGE_PATH, fname)
    img = np.array(Image.open(img_path).convert("L")).flatten()
    images.append(img)
    # label
    labels.append(row['label'])
    # bboxes
    bboxes.append((row['min_x'], row['min_y'], row['width'], row['height']))
images = np.array(images)
bboxes = np.array(bboxes)

# free memory
del df
del df_new

Total Samples: 4289
Samples with bbox: 4289


100%|██████████| 4289/4289 [00:02<00:00, 1599.20it/s]


In [51]:
frac_train = 0.7
frac_val = 0.15
frac_test = 0.15

label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

features = np.concatenate((images, labels_encoded.reshape(-1, 1)), axis=1)

X_train, X_tmp, Y_train, Y_tmp = train_test_split(features, bboxes, test_size=frac_val+frac_test)
X_valid, X_test, Y_valid, Y_test = train_test_split(X_tmp, Y_tmp, test_size=frac_val/(frac_val+frac_test))

print(f"Number of Training: {len(Y_train)}")
print(f"Number of Validation: {len(Y_valid)}")
print(f"Number of Testing: {len(Y_test)}")

Number of Training: 3002
Number of Validation: 643
Number of Testing: 644


In [52]:
import numpy as np

# Helper function to calculate IoU for two bounding boxes
def calculate_iou(bbox1, bbox2):
    # Convert (min_x, min_y, width, height) to (x1, y1, x2, y2)
    bbox1_x1, bbox1_y1, bbox1_w, bbox1_h = bbox1
    bbox1_x2, bbox1_y2 = bbox1_x1 + bbox1_w, bbox1_y1 + bbox1_h

    bbox2_x1, bbox2_y1, bbox2_w, bbox2_h = bbox2
    bbox2_x2, bbox2_y2 = bbox2_x1 + bbox2_w, bbox2_y1 + bbox2_h

    # Determine the coordinates of the intersection rectangle
    x_overlap_min = max(bbox1_x1, bbox2_x1)
    y_overlap_min = max(bbox1_y1, bbox2_y1)
    x_overlap_max = min(bbox1_x2, bbox2_x2)
    y_overlap_max = min(bbox1_y2, bbox2_y2)

    # Calculate the area of intersection
    intersection_width = max(0, x_overlap_max - x_overlap_min)
    intersection_height = max(0, y_overlap_max - y_overlap_min)
    intersection_area = intersection_width * intersection_height

    # Calculate the area of both bounding boxes
    area1 = bbox1_w * bbox1_h
    area2 = bbox2_w * bbox2_h

    # Calculate the area of union
    union_area = area1 + area2 - intersection_area

    # Handle division by zero
    if union_area == 0:
        return 0.0

    # Calculate IoU
    iou = intersection_area / union_area
    return iou

def get_mean_iou_score(Y_true, Y_pred):
    # Calculate IoU for each pair of bounding boxes and return the mean
    ious = [calculate_iou(y_true, y_pred) for y_true, y_pred in zip(Y_true, Y_pred)]
    return np.mean(ious)

def get_iou_accuracy(Y_true, Y_pred, threshold=0.7):
    correct_predictions = 0
    total_predictions = len(Y_true)
    for y_true, y_pred in zip(Y_true, Y_pred):
        iou = calculate_iou(y_true, y_pred)
        if iou > threshold:
            correct_predictions += 1
    return correct_predictions / total_predictions if total_predictions > 0 else 0.0

In [ ]:
# hyperparameters to try
hyp_n_estimators = [60, 80, 100, 120, 140]
max_depths = [10, 15, 30]

val_results = [] # Store (n_est, max_depth, mean_iou, iou_accuracy)

for n_est in tqdm(hyp_n_estimators, position=0, desc='n_est', leave=True):
  for max_depth in tqdm(max_depths, position=1, desc='max_depth', leave=False):
    rfc = RandomForestRegressor(n_estimators=n_est, random_state=seed,
                                 max_depth=max_depth, n_jobs=-1)
    rfc.fit(X_train, Y_train)
    Y_valid_pred = rfc.predict(X_valid)
    mean_iou = get_mean_iou_score(Y_valid, Y_valid_pred)
    iou_accuracy = get_iou_accuracy(Y_valid, Y_valid_pred, threshold=0.7)
    val_results.append((n_est, max_depth, mean_iou, iou_accuracy))
    print(f"n_estimators: {n_est}, max_depth: {max_depth}, mean_iou: {mean_iou:.4f}, iou_accuracy (>0.7): {iou_accuracy:.4f}")

In [ ]:
# fit best regressor)
rfc = RandomForestRegressor(n_estimators=140, random_state=seed,
                              max_depth=30, n_jobs=-1)
rfc.fit(X_train, Y_train)
Y_test_pred = rfc.predict(X_test)
mean_iou = get_mean_iou_score(Y_test, Y_test_pred)
iou_accuracy = get_iou_accuracy(Y_test, Y_test_pred, threshold=0.7)
print(f"Test Mean IoU: {mean_iou:.4f}")
print(f"Test IoU Accuracy (>0.7): {iou_accuracy:.4f}")